# The splits that are already in the folder

`u.data` is the complete set of 100000 ratings, in random order. The other rating files are cuts of that same set.

`ua.test` holds 10 ratings for each of the 943 users, and `ua.base` holds everything else. `ub.test` also holds 10 ratings per user, and it shares no rating with `ua.test`. A result on `ua` is one experiment. Repeating it on `ub` is a second experiment, not a second look at the same hidden ratings.

`u1.test` through `u5.test` each hold 20000 ratings, and `u1.base` through `u5.base` each hold 80000. The five test sets are disjoint, and their union is the whole 100000. That is the five-fold cut described in the README. Averaging five RMSEs, one from each fold, is a different protocol from reporting the single `ua` number.

This path reports the `ua` number, because the "ten ratings per person" rule is easy to check: $943 \times 10 = 9430$. The scripts `mku.sh` and `allbut.pl` can rebuild these cuts from `u.data`. The lessons read the cuts that are already written.

The timestamps on the whole file run from 874724710 to 893286638. In UTC those are 20 September 1997, 03:05:10, and 22 April 1998, 23:10:38. The README dates the collection from 19 September 1997 through 22 April 1998. The earliest stamp falls on 20 September in UTC, which is still 19 September in the evening in Minnesota.


In [2]:
# Locate the MovieLens tables beside this notebook, or under the course root.
from pathlib import Path

def data_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "16-Movie Recommendation" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "u.info").exists():
            return direct
    raise FileNotFoundError(name)

def read_lines(name):
    return data_path(name).read_text(encoding="latin-1").splitlines()

def read_ratings(name):
    rows = []
    for line in read_lines(name):
        if not line.strip():
            continue
        user, item, rating, stamp = line.split("\t")
        rows.append((int(user), int(item), int(rating), int(stamp)))
    return rows

ratings = read_ratings("u.data")
print(len(ratings), ratings[0])


100000 (196, 242, 3, 881250949)


In [3]:
# Ten per user, two disjoint tests, and five disjoint folds.
from collections import Counter
from datetime import datetime, timezone
ua_test = read_ratings("ua.test")
ub_test = read_ratings("ub.test")
print(Counter(Counter(user for user, _, _, _ in ua_test).values()))
assert Counter(user for user, _, _, _ in ua_test) and set(
    Counter(user for user, _, _, _ in ua_test).values()
) == {10}
assert len(ua_test) == 9430
ua_keys = {(user, item) for user, item, _, _ in ua_test}
ub_keys = {(user, item) for user, item, _, _ in ub_test}
assert ua_keys.isdisjoint(ub_keys)
folds = []
for index in range(1, 6):
    test = read_ratings(f"u{index}.test")
    base = read_ratings(f"u{index}.base")
    print(index, len(test), len(base))
    assert (len(test), len(base)) == (20000, 80000)
    folds.append({(user, item) for user, item, _, _ in test})
assert all(folds[i].isdisjoint(folds[j]) for i in range(5) for j in range(i))
assert set().union(*folds) == {(user, item) for user, item, _, _ in ratings}
stamps = [stamp for _, _, _, stamp in ratings]
earliest = datetime.fromtimestamp(min(stamps), timezone.utc)
latest = datetime.fromtimestamp(max(stamps), timezone.utc)
print(min(stamps), earliest.isoformat(), max(stamps), latest.isoformat())
assert min(stamps) == 874724710
assert max(stamps) == 893286638
assert earliest.isoformat() == "1997-09-20T03:05:10+00:00"
assert latest.isoformat() == "1998-04-22T23:10:38+00:00"


Counter({10: 943})
1 20000 80000
2 20000 80000
3 20000 80000
4 20000 80000
5 20000 80000
874724710 1997-09-20T03:05:10+00:00 893286638 1998-04-22T23:10:38+00:00


## Pitfall

Training on `u.data` and then scoring `ua.test` uses the test ratings twice: they are inside `u.data`. The holdout lesson trains on `ua.base` only. The five folds are a separate protocol, and this path does not average them into the reported error.

## Summary

`ua.test` is 10 ratings for each of 943 users. `ub.test` is another such cut, disjoint from it. The five `u1`–`u5` test files are disjoint 20000-rating folds whose union is the whole set.
